Structured output refers to the process of forcing a Large Language Model (LLM) to return data in a specific, predictable format (like JSON) rather than plain conversational text.

### Why It Matters

* **Programmatic Integration:** Code can’t easily "read" a paragraph, but it can parse a JSON object to trigger functions or update databases.
* **Reliability:** It ensures the model provides all required fields (e.g., always including a "price" and "currency").
* **Data Validation:** Frameworks like Pydantic can automatically check if the "age" field is an integer or if an "email" is valid.

### 1. Using Pydantic (Recommended)

Pydantic is the gold standard for structured data in Python. It provides built-in validation and type hints.
* **Highlight:** **Strict Validation.** If the model returns a string for "age," Pydantic will throw an error or attempt to coerce it, ensuring your downstream code doesn't crash.

In [19]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field


class Person(BaseModel):
    name: str = Field(description="The person's full name")
    age: int = Field(description="The person's age in years")


llm = ChatOllama(model="gpt-oss:20b")
structured_llm = llm.with_structured_output(Person)
response = structured_llm.invoke("John Doe is 30 years old.")
print(response)

name='John Doe' age=30


### 2. Using TypedDict

`TypedDict` is a lighter, built-in Python alternative. It provides type hints for dictionaries but does **not** perform runtime validation.
* **Highlight:** **Zero Overhead.** Great for simple data structures where you don't need the heavy features of Pydantic and want to keep dependencies low.

In [20]:
from typing import TypedDict
from langchain_ollama import ChatOllama


class Movie(TypedDict):
    title: str
    year: int


llm = ChatOllama(model="gpt-oss:20b")
structured_llm = llm.with_structured_output(Movie)
response = structured_llm.invoke("Inception was released in 2010.")
print(response)

{'title': 'Inception', 'year': 2010}


### 3. Using JSON Schema (Raw)

This involves passing a standard JSON Schema dictionary. This is how the "under the hood" communication often happens with APIs.
* **Highlight:** **Language Agnostic.** JSON Schema is a universal standard. Use this if you are building schemas that need to be shared across different programming languages (e.g., Python and JavaScript).

In [21]:
from langchain_ollama import ChatOllama

json_schema = {
    "title": "joke_schema",
    "type": "object",
    "properties": {"setup": {"type": "string"}, "punchline": {"type": "string"}},
    "required": ["setup", "punchline"],
}

llm = ChatOllama(model="gpt-oss:20b")
structured_llm = llm.with_structured_output(json_schema)

response = structured_llm.invoke("Tell me a joke about robots.")
print(response)

{'setup': 'Why did the robot take a break from its job?', 'punchline': 'It needed to *reboot* its sense of humor!'}
